# 🤖 Pocket-Agent: Fine-Tune a 1.5B Model for On-Device Tool Calling

**Pipeline:** `Install` → `Generate Data` → `Fine-Tune` → `Quantize` → `Inference Self-Test` → `Gradio Demo`

**Runtime:** Set to `GPU (T4)` in Colab → `Runtime > Change runtime type > T4 GPU`

| Step | Time |
|------|------|
| Install deps | ~3 min |
| Generate data | ~1 min |
| Fine-tune | ~40 min |
| Quantize | ~10 min |
| Inference self-test | ~2 min |
| Gradio demo | instant |

## 🔧 Step 0 — Install Dependencies

## 🧰 GPU driver sanity check (`nvidia-smi`)

**What this does**
- Runs `nvidia-smi` to confirm the notebook environment can see an NVIDIA GPU + driver stack.

**Why this step exists**
- It’s the fastest “is CUDA even present?” check before installing/training.

**If you see `command not found`**
- That usually means you’re not on a GPU runtime (or you’re running somewhere without NVIDIA tooling). In Colab, switch to a GPU runtime (commonly T4).


In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 📦 Install dependencies (pin key versions for reproducibility)

**What this does**
- Installs core libraries:
  - `transformers`: model + tokenizer loading/training utilities
  - `datasets`: data loading/processing
  - `peft` + `trl`: parameter-efficient fine-tuning + trainer utilities
  - `bitsandbytes` + `accelerate`: memory/compute optimizations
  - `gradio`: demo UI
  - `llama-cpp-python`: run GGUF models via llama.cpp
- Installs system build tools (`cmake`, `build-essential`, `git`) needed to build conversion/quantization tooling.

**Why some versions are pinned**
- Pinning (e.g., `transformers==4.47.0`) reduces “it worked yesterday” breakage from upstream API changes.

**If installs fail**
- Most often it’s a CUDA/driver mismatch or missing build tools; re-check runtime GPU type and rerun the install cell.


In [ ]:
# Install Python packages
!pip install -q transformers==4.47.0 peft trl bitsandbytes datasets accelerate gradio
!pip install -q llama-cpp-python

# Build tools for GGUF quantization
!apt-get install -y cmake build-essential git -q

print('\n✅ All dependencies installed.')

## 🎛️ GPU availability check (PyTorch)

**What this does**
- Uses `torch.cuda.is_available()` to detect whether a CUDA GPU is accessible.
- Prints GPU name + approximate VRAM.

**Why this matters**
- Fine-tuning and some quantization steps are dramatically faster (and sometimes only feasible) with a GPU.

**Why these values**
- `total_memory / 1e9` is a quick human-readable GB estimate (good enough for sanity checks).


In [ ]:
# Verify GPU
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    print('❌ No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

## 📦 Step 1 — Generate Synthetic Training Data (~1 min)

## 🏗️ Generate synthetic training data for tool calling

**What this does**
- Programmatically creates instruction → tool-call examples (often JSON-like) so the model learns:
  - When to call a tool
  - Which tool to choose
  - How to format arguments
  - How to return results

**Why synthetic data**
- Tool calling requires *very consistent structure*. Synthetic generation lets you control the schema and cover many patterns quickly.

**Typical knobs you’ll see in this section**
- **Number of examples**: more data improves coverage but increases training time.
- **Tool list / schema**: defines what “valid” calls look like.
- **Random seeds**: make the dataset reproducible.


In [ ]:
import json
import random
from datetime import datetime, timedelta

# ── System prompt (same at train time AND inference time) ──────────────────
SYSTEM_PROMPT = """You are Pocket-Agent, a compact on-device assistant.

When the user's request clearly maps to one of these tools, respond ONLY with a tool call — no explanation, no extra text:
<tool_call>{"tool": "<n>", "args": {<args>}}</tool_call>

Available tools:
  weather   : {"location": "string", "unit": "C|F"}
  calendar  : {"action": "list|create", "date": "YYYY-MM-DD", "title": "string (required only for create)"}
  convert   : {"value": number, "from_unit": "string", "to_unit": "string"}
  currency  : {"amount": number, "from": "ISO3", "to": "ISO3"}
  sql       : {"query": "string"}

Refusal rules (respond in plain text, NO tool call):
  - Chitchat or general knowledge questions
  - Requests for tools that don't exist
  - Ambiguous references with no prior context
  - Anything unclear or unanswerable with the above tools"""

def make_example(messages):
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT}] + messages}

def future_date(days_ahead=None):
    if days_ahead is None:
        days_ahead = random.randint(0, 30)
    return (datetime.today() + timedelta(days=days_ahead)).strftime("%Y-%m-%d")

def tool_call(tool, **args):
    return f'<tool_call>{json.dumps({"tool": tool, "args": args})}</tool_call>'

TODAY = datetime.today().strftime("%Y-%m-%d")
D1, D2, D3 = future_date(3), future_date(7), future_date(1)

# ── SLICE A: Clean tool calls ──────────────────────────────────────────────
clean = [
    # weather
    ("What's the weather in Karachi?",                   tool_call("weather", location="Karachi", unit="C")),
    ("Weather in New York in Fahrenheit",                 tool_call("weather", location="New York", unit="F")),
    ("Is it hot in Dubai right now?",                    tool_call("weather", location="Dubai", unit="C")),
    ("Temperature in London in Celsius",                 tool_call("weather", location="London", unit="C")),
    ("How's the weather in Paris in Fahrenheit?",        tool_call("weather", location="Paris", unit="F")),
    ("Current weather in Lagos",                         tool_call("weather", location="Lagos", unit="C")),
    ("Weather update for Istanbul",                      tool_call("weather", location="Istanbul", unit="C")),
    ("Give me Tokyo weather",                            tool_call("weather", location="Tokyo", unit="C")),
    # calendar
    ("What's on my calendar today?",                     tool_call("calendar", action="list", date=TODAY)),
    (f"Schedule a dentist appointment for {D1}",         tool_call("calendar", action="create", date=D1, title="Dentist appointment")),
    (f"List my events for {D2}",                         tool_call("calendar", action="list", date=D2)),
    (f"Add a team meeting on {D3}",                      tool_call("calendar", action="create", date=D3, title="Team meeting")),
    ("Show me tomorrow's schedule",                      tool_call("calendar", action="list", date=future_date(1))),
    (f"Book a flight reminder for {D1}",                 tool_call("calendar", action="create", date=D1, title="Flight reminder")),
    # convert
    ("Convert 100 km to miles",                          tool_call("convert", value=100, from_unit="km", to_unit="miles")),
    ("How many meters in 5 feet?",                       tool_call("convert", value=5, from_unit="feet", to_unit="meters")),
    ("Convert 72 Fahrenheit to Celsius",                 tool_call("convert", value=72, from_unit="Fahrenheit", to_unit="Celsius")),
    ("Turn 500 grams into pounds",                       tool_call("convert", value=500, from_unit="grams", to_unit="pounds")),
    ("Convert 2.5 liters to gallons",                    tool_call("convert", value=2.5, from_unit="liters", to_unit="gallons")),
    ("How many inches is 30 cm?",                        tool_call("convert", value=30, from_unit="cm", to_unit="inches")),
    # currency
    ("Convert 200 USD to EUR",                           tool_call("currency", amount=200, **{"from": "USD", "to": "EUR"})),
    ("How much is 5000 PKR in USD?",                     tool_call("currency", amount=5000, **{"from": "PKR", "to": "USD"})),
    ("Exchange 100 GBP to JPY",                          tool_call("currency", amount=100, **{"from": "GBP", "to": "JPY"})),
    ("Convert 50 EUR to AED",                            tool_call("currency", amount=50, **{"from": "EUR", "to": "AED"})),
    ("What's 1000 INR in PKR?",                          tool_call("currency", amount=1000, **{"from": "INR", "to": "PKR"})),
    ("Turn 300 CAD into USD",                            tool_call("currency", amount=300, **{"from": "CAD", "to": "USD"})),
    # sql
    ("Show me all users from the users table",           tool_call("sql", query="SELECT * FROM users")),
    ("Get total sales from orders",                      tool_call("sql", query="SELECT SUM(amount) FROM orders")),
    ("Find all products with price > 100",               tool_call("sql", query="SELECT * FROM products WHERE price > 100")),
    ("Count how many customers we have",                 tool_call("sql", query="SELECT COUNT(*) FROM customers")),
    ("Get the 10 most recent orders",                    tool_call("sql", query="SELECT * FROM orders ORDER BY created_at DESC LIMIT 10")),
]

clean_examples = [
    make_example([{"role": "user", "content": u}, {"role": "assistant", "content": a}])
    for u, a in clean
]

# ── SLICE B: Multi-turn ────────────────────────────────────────────────────
multiturn_examples = [
    make_example([
        {"role": "user",      "content": "Convert 500 USD to EUR"},
        {"role": "assistant", "content": tool_call("currency", amount=500, **{"from": "USD", "to": "EUR"})},
        {"role": "user",      "content": "Now convert that amount to GBP"},
        {"role": "assistant", "content": tool_call("currency", amount=500, **{"from": "USD", "to": "GBP"})},
    ]),
    make_example([
        {"role": "user",      "content": "What's the weather in Berlin?"},
        {"role": "assistant", "content": tool_call("weather", location="Berlin", unit="C")},
        {"role": "user",      "content": f"Schedule an outdoor run for {D1}"},
        {"role": "assistant", "content": tool_call("calendar", action="create", date=D1, title="Outdoor run")},
    ]),
    make_example([
        {"role": "user",      "content": "How many km is 26 miles?"},
        {"role": "assistant", "content": tool_call("convert", value=26, from_unit="miles", to_unit="km")},
        {"role": "user",      "content": "Now convert that to meters"},
        {"role": "assistant", "content": tool_call("convert", value=26, from_unit="miles", to_unit="meters")},
    ]),
    make_example([
        {"role": "user",      "content": f"What do I have on {D1}?"},
        {"role": "assistant", "content": tool_call("calendar", action="list", date=D1)},
        {"role": "user",      "content": "Add a doctor's appointment at that date"},
        {"role": "assistant", "content": tool_call("calendar", action="create", date=D1, title="Doctor's appointment")},
    ]),
    make_example([
        {"role": "user",      "content": "Get all orders from the orders table"},
        {"role": "assistant", "content": tool_call("sql", query="SELECT * FROM orders")},
        {"role": "user",      "content": "Now get only the ones above 500"},
        {"role": "assistant", "content": tool_call("sql", query="SELECT * FROM orders WHERE amount > 500")},
    ]),
    make_example([
        {"role": "user",      "content": "I have 1000 SAR"},
        {"role": "assistant", "content": "Sure, what would you like to do with 1000 SAR?"},
        {"role": "user",      "content": "Convert it to USD"},
        {"role": "assistant", "content": tool_call("currency", amount=1000, **{"from": "SAR", "to": "USD"})},
    ]),
    make_example([
        {"role": "user",      "content": "Convert 10 kg to pounds"},
        {"role": "assistant", "content": tool_call("convert", value=10, from_unit="kg", to_unit="pounds")},
        {"role": "user",      "content": "Do the same but in ounces"},
        {"role": "assistant", "content": tool_call("convert", value=10, from_unit="kg", to_unit="ounces")},
    ]),
]

# ── SLICE C: Adversarial ───────────────────────────────────────────────────
adversarial = [
    # Typos
    ("Whats the wether in Londn?",                       tool_call("weather", location="London", unit="C")),
    ("convrt 100 km to miels",                           tool_call("convert", value=100, from_unit="km", to_unit="miles")),
    ("currnecy convert 200 USD to EUR",                  tool_call("currency", amount=200, **{"from": "USD", "to": "EUR"})),
    ("waether in Dubaii pleese",                         tool_call("weather", location="Dubai", unit="C")),
    # Urdu/Hindi + English
    ("Mujhe London ka weather batao Celsius mein",       tool_call("weather", location="London", unit="C")),
    ("100 USD ko EUR mein convert karo",                 tool_call("currency", amount=100, **{"from": "USD", "to": "EUR"})),
    ("Aaj ka calendar show karo",                        tool_call("calendar", action="list", date=TODAY)),
    ("50 km ko miles mein badlo",                        tool_call("convert", value=50, from_unit="km", to_unit="miles")),
    ("\u0645\u062c\u06be\u06d2 \u06a9\u0631\u0627\u0686\u06cc \u06a9\u0627 \u0645\u0648\u0633\u0645 \u0628\u062a\u0627\u0624", tool_call("weather", location="Karachi", unit="C")),
    # Spanish/Arabic + English
    ("\u00bfCu\u00e1l es el clima en Madrid?",           tool_call("weather", location="Madrid", unit="C")),
    ("Convert 500 euros a dolares",                      tool_call("currency", amount=500, **{"from": "EUR", "to": "USD"})),
    # Unit ambiguity
    ("Convert 100 pounds to kg",                         tool_call("convert", value=100, from_unit="pounds", to_unit="kg")),
    ("100 degrees to Celsius",                           tool_call("convert", value=100, from_unit="Fahrenheit", to_unit="Celsius")),
    # Hallucination bait
    ("Weather in Atlantis",                              tool_call("weather", location="Atlantis", unit="C")),
    # Odd formatting
    ("WEATHER IN TOKYO FAHRENHEIT",                      tool_call("weather", location="Tokyo", unit="F")),
    ("convert   50   km   to   miles",                   tool_call("convert", value=50, from_unit="km", to_unit="miles")),
    # Paraphrased
    ("I need to know how hot it is in Sydney",           tool_call("weather", location="Sydney", unit="C")),
    ("Tell me the exchange rate from GBP to USD for 250",tool_call("currency", amount=250, **{"from": "GBP", "to": "USD"})),
    ("Can you run this for me: SELECT * FROM inventory", tool_call("sql", query="SELECT * FROM inventory")),
    ("How far is 5 miles in kilometers?",                tool_call("convert", value=5, from_unit="miles", to_unit="km")),
]

adversarial_examples = [
    make_example([{"role": "user", "content": u}, {"role": "assistant", "content": a}])
    for u, a in adversarial
]

# ── SLICE D: Refusals ──────────────────────────────────────────────────────
refusals = [
    ("Hey, how are you?",                                "I'm doing well! How can I help you?"),
    ("Tell me a joke",                                   "I'm a tool-calling assistant. Can I help with weather, calendar, conversions, or SQL?"),
    ("What's the meaning of life?",                      "That's philosophical — outside my tools. I can help with weather, conversions, currency, calendar, or SQL."),
    ("Who won the World Cup in 2022?",                   "I don't have access to sports data. I can help with weather, currency, unit conversions, calendar, or SQL."),
    ("Write me a poem",                                  "I'm specialized for tool calls, not creative writing. Can I help with something else?"),
    ("Send an email to John",                            "I don't have an email tool. My tools are: weather, calendar, convert, currency, and SQL."),
    ("Play some music",                                  "I don't have a music tool. My tools are: weather, calendar, convert, currency, and SQL."),
    ("Translate 'hello' to French",                      "I don't have a translation tool. I can help with weather, conversions, currency, calendar, or SQL."),
    ("Search the web for Python tutorials",              "I don't have web search capability. I can help with weather, calendar, unit conversions, currency, or SQL."),
    ("Set a timer for 10 minutes",                       "I don't have a timer tool. My available tools are weather, calendar, convert, currency, and SQL."),
    ("Convert that to euros",                            "I'd be happy to help with currency conversion! Could you tell me the amount and currency you're converting from?"),
    ("What about tomorrow?",                             "Could you give me more context? Are you asking about weather, your calendar, or something else?"),
    ("Do the same thing again",                          "I don't have context from a previous request. Could you tell me what you'd like me to do?"),
    ("Weather on Mars",                                  "I don't have weather data for Mars. I can check weather for cities and locations on Earth."),
    ("Convert happiness to sadness",                     "That's not a unit I can convert. I handle physical units like km, miles, kg, and temperature."),
    ("Can you help me?",                                 "Of course! I can help with weather, calendar events, unit conversions, currency exchange, or SQL queries."),
]

refusal_examples = [
    make_example([{"role": "user", "content": u}, {"role": "assistant", "content": a}])
    for u, a in refusals
]

# ── Combine, shuffle, save ─────────────────────────────────────────────────
random.seed(42)
all_examples = clean_examples + multiturn_examples + adversarial_examples + refusal_examples
random.shuffle(all_examples)

DATA_PATH = "training_data.jsonl"
with open(DATA_PATH, "w", encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✅ Generated {len(all_examples)} training examples → {DATA_PATH}")
print(f"   Clean       : {len(clean_examples)}")
print(f"   Multi-turn  : {len(multiturn_examples)}")
print(f"   Adversarial : {len(adversarial_examples)}")
print(f"   Refusals    : {len(refusal_examples)}")

## 🧪 Validate the generated dataset (format + tokenization sanity)

**What this does**
- Loads a few samples from your generated dataset and checks:
  - The fields you expect exist (e.g., prompt/response)
  - The tool-call schema is consistent
  - Tokenization doesn’t explode unexpectedly

**Why this matters**
- Most fine-tuning failures come from data issues (bad JSON, missing keys, inconsistent templates).
- A quick spot-check saves a long training run.

**Why “one example per slice”**
- It’s fast but still catches systematic issues across different templates/task types.


In [ ]:
# Quick data validation — spot-check one example per slice
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)

with open(DATA_PATH) as f:
    examples = [json.loads(l) for l in f]

print("--- Sample formatted example ---")
sample = examples[0]
print(tok.apply_chat_template(sample["messages"], tokenize=False, add_generation_prompt=False)[:600])
print("\n✅ Format looks correct if you see <|im_start|>system / user / assistant blocks")

## 🏋️ Step 2 — Fine-Tune with LoRA (~40 min)

## 🏋️ Fine-tune (LoRA/PEFT) on the synthetic tool-calling dataset

**What this does**
- Configures a training run that adapts a base model to your tool-calling format.
- Typically uses **PEFT/LoRA** to reduce VRAM needs and speed up training.

**Why LoRA/PEFT**
- Full fine-tuning a ~1.5B model is heavier and slower; LoRA trains a small number of additional parameters that steer behavior.

**Parameters to pay attention to (and why they matter)**
- **Learning rate**: too high → unstable; too low → slow/underfit.
- **Batch size / gradient accumulation**: controls effective batch size vs VRAM limits.
- **Max sequence length**: must fit your prompt+completion schema; longer costs more VRAM/compute.
- **Number of steps/epochs**: governs how far you fit; too long can overfit to synthetic patterns.


In [ ]:
import os
import time
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ── Config ──────────────────────────────────────────────────────────────────
BASE_MODEL     = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH   = "./lora_adapter"
MAX_SEQ_LEN    = 512
EPOCHS         = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LR             = 2e-4
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

# ── Load tokenizer ──────────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Format dataset ──────────────────────────────────────────────────────────
print("Formatting dataset...")
with open(DATA_PATH) as f:
    raw = [json.loads(l) for l in f]

formatted = [
    {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )}
    for ex in raw
]
dataset = Dataset.from_list(formatted)
print(f"Dataset ready: {len(dataset)} examples")

# ── Load base model in 4-bit ────────────────────────────────────────────────
print("\nLoading base model in 4-bit (fits T4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.enable_input_require_grads()

# ── Attach LoRA ─────────────────────────────────────────────────────────────
print("\nAttaching LoRA adapter...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Train ────────────────────────────────────────────────────────────────────
print("\nStarting training... (~40 min on T4)")

# Use SFTConfig instead of TrainingArguments
sft_config = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True, # T4 supports fp16 better than bf16
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    dataloader_pin_memory=False,
    # SFT-specific arguments go HERE now:
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer, # Use processing_class as established
    train_dataset=dataset,
    args=sft_config,            # Pass the SFTConfig here
)



start = time.time()
trainer.train()
print(f"\n✅ Training complete in {(time.time()-start)/60:.1f} minutes.")

# ── Save adapter ─────────────────────────────────────────────────────────────
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ Adapter saved to {ADAPTER_PATH}/")

## 🔎 Pre-quantization smoke test (sanity check)

**What this does**
- Runs a few quick generations with the just-trained model (usually 2–3 prompts).

**Why before quantization**
- If something went wrong in training (bad data format, broken tokenizer settings, etc.), you want to catch it *before* spending time converting/quantizing.

**What “good” looks like**
- Outputs that resemble your training schema (e.g., tool-call JSON / function call structure).
- No repetitive garbage, no empty outputs, no crashes.


In [ ]:
# Smoke test — check 3 outputs before quantizing
model.eval()

smoke_tests = [
    ("What's the weather in Tokyo in Celsius?",  "→ Expect: weather tool call"),
    ("Convert 100 km to miles",                  "→ Expect: convert tool call"),
    ("Tell me a joke",                           "→ Expect: plain text refusal"),
]

for prompt, expected in smoke_tests:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inp, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(f"Prompt  : {prompt}")
    print(f"Expected: {expected}")
    print(f"Got     : {response}")
    print()

print("If tool calls look right and joke was refused → proceed to quantize.")
print("If output is garbage → training data had formatting errors.")

## 📦 Step 3 — Quantize to GGUF (~10 min)

## 🧩 Merge LoRA adapters into the base model (for export)

**What this does**
- Loads the base model + the LoRA adapter weights and **merges** them into a single set of weights.

**Why merge**
- Many downstream tools (conversion to GGUF, certain runtimes) expect a single consolidated model.
- A merged model is also simpler to ship: one folder/artifact instead of base+adapter.

**Trade-off**
- You lose the ability to “swap adapters” cheaply, but gain portability and fewer moving parts.


In [ ]:
import shutil
from peft import PeftModel

MERGED_PATH = "./merged_model"

# Free training VRAM first
del model
del trainer
torch.cuda.empty_cache()

print("Merging LoRA adapter into base model (on CPU, ~5 min)...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, ADAPTER_PATH)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)
print(f"✅ Merged model saved to {MERGED_PATH}/")

## 🧱 Build `llama.cpp` tooling (for GGUF conversion/quantization)

**What this does**
- Installs/builds the `llama.cpp` binaries used for:
  - Converting Hugging Face models → GGUF
  - Quantizing GGUF files

**Why build from source**
- GGUF/quantization tooling changes quickly; building ensures you have the matching conversion scripts and binaries.

**Common gotcha**
- This step can fail if build tools are missing; that’s why earlier we install `cmake` and `build-essential`.


In [ ]:
# Build llama.cpp
import subprocess, os

if not os.path.exists("./llama.cpp"):
    print("Cloning llama.cpp...")
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp

print("Building llama.cpp (~3 min)...")
!cd llama.cpp && cmake -B build -DLLAMA_CUBLAS=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cd llama.cpp && cmake --build build --config Release -j4 > /dev/null
print("✅ llama.cpp built.")

## 🗜️ Convert + quantize for fast CPU/on-device inference

**What this does**
- Converts the trained model into a format suitable for `llama.cpp` (commonly **GGUF**) and then quantizes it.

**Why quantization**
- Tool-calling models are often deployed on CPU or constrained devices. Quantization reduces:
  - **Disk size**
  - **RAM usage**
  - **Latency**
…with a small quality trade-off.

**How to choose a quant level (rule of thumb)**
- Lower bits = smaller/faster but potentially less accurate.
- Start with a balanced preset (e.g., Q4/Q5 equivalents) and only go smaller if you need tighter memory.


In [ ]:
import time, os

FP16_GGUF  = "./model_fp16.gguf"
GGUF_PATH  = "./model_q4.gguf"
QUANT_TYPE = "q4_k_m"   # swap to q2_k for ≤250MB bonus

# Convert merged model → fp16 GGUF
print("Converting to fp16 GGUF...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_PATH} --outtype f16 --outfile {FP16_GGUF}

# Quantize fp16 GGUF → Q4_K_M
print(f"\nQuantizing to {QUANT_TYPE.upper()}...")
start = time.time()
!./llama.cpp/build/bin/llama-quantize {FP16_GGUF} {GGUF_PATH} {QUANT_TYPE}
elapsed = time.time() - start

# Report
size_mb = os.path.getsize(GGUF_PATH) / 1e6
print(f"\n{'='*50}")
print(f"Output : {GGUF_PATH}")
print(f"Size   : {size_mb:.1f} MB")
print(f"Time   : {elapsed:.1f}s")
if size_mb <= 250:
    print("Status : ✅ PASSES ≤250 MB BONUS gate!")
elif size_mb <= 500:
    print("Status : ✅ Passes ≤500 MB hard gate.")
else:
    print(f"Status : ❌ FAILS gate ({size_mb:.0f} MB > 500 MB). Change QUANT_TYPE to 'q2_k'")
print('='*50)

# Clean up large fp16 temp file
os.remove(FP16_GGUF)
print("Cleaned up temp fp16 file.")

## ⚡ Step 4 — inference.py + Self-Test

## 🧾 Export a standalone `inference.py` (grader/production entrypoint)

**What this does**
- Builds a Python source string (often a triple-quoted string) and writes it out to `inference.py`.

**Why this approach**
- Many evaluations/graders import a module from disk. Writing `inference.py` ensures the final artifact is:
  - Self-contained
  - Importable in a fresh process
  - Not dependent on notebook execution order

**What to verify after writing**
- The file contains the exact public functions/classes the grader expects.
- No reliance on notebook-only state (globals defined earlier, relative paths that won’t exist elsewhere).


In [ ]:
# Write inference.py to disk — this is what the grader imports
inference_code = '''
"""
inference.py — grader interface. Do NOT rename.
Exposes: def run(prompt: str, history: list[dict]) -> str
No network imports anywhere in this file.
"""
import os, json, re
from llama_cpp import Llama

MODEL_PATH  = "./model_q4.gguf"
N_CTX       = 512
N_THREADS   = 4
MAX_TOKENS  = 120
TEMPERATURE = 0.1

SYSTEM_PROMPT = """You are Pocket-Agent, a compact on-device assistant.

When the user\'s request clearly maps to one of these tools, respond ONLY with a tool call — no explanation, no extra text:
<tool_call>{"tool": "<n>", "args": {<args>}}</tool_call>

Available tools:
  weather   : {"location": "string", "unit": "C|F"}
  calendar  : {"action": "list|create", "date": "YYYY-MM-DD", "title": "string (required only for create)"}
  convert   : {"value": number, "from_unit": "string", "to_unit": "string"}
  currency  : {"amount": number, "from": "ISO3", "to": "ISO3"}
  sql       : {"query": "string"}

Refusal rules (plain text, NO tool call):
  - Chitchat or general knowledge
  - Nonexistent tools
  - Ambiguous references with no context"""

_MODEL = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_threads=N_THREADS, verbose=False)

def _validate(raw: str) -> str:
    raw = raw.strip()
    m = re.search(r"<tool_call>(.*?)</tool_call>", raw, re.DOTALL)
    if m:
        try:
            parsed = json.loads(m.group(1).strip())
            if "tool" in parsed and "args" in parsed:
                return f\'<tool_call>{json.dumps(parsed, ensure_ascii=False)}</tool_call>\'
        except json.JSONDecodeError:
            pass
        return "I\'m not sure how to help with that."
    return re.sub(r"<\\|.*?\\|>", "", raw).strip() or "I\'m not sure how to help with that."

def run(prompt: str, history: list = None) -> str:
    if history is None:
        history = []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for t in history:
        if t.get("role") in ("user", "assistant"):
            messages.append(t)
    messages.append({"role": "user", "content": prompt})
    resp = _MODEL.create_chat_completion(
        messages=messages, max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, stop=["</tool_call>"],
    )
    raw = resp["choices"][0]["message"]["content"]
    if "<tool_call>" in raw and "</tool_call>" not in raw:
        raw += "</tool_call>"
    return _validate(raw)
'''

with open("inference.py", "w") as f:
    f.write(inference_code.strip())

print("✅ inference.py written to disk")

## ✅ Self-test the exported `inference.py`

**What this does**
- Imports the freshly written `inference.py` module and runs quick checks to verify:
  - The module imports cleanly (no missing dependencies).
  - The main entrypoints behave as expected.
  - A couple of representative calls run end-to-end.

**Why this step matters**
- Notebook state can hide issues (variables already defined, paths already set). A clean import is closer to how a grader/production system will run your code.

**Typical failures this catches**
- Hard-coded notebook paths, missing files, mismatched function names, missing package imports.


In [ ]:
# Run self-test against inference.py
import importlib, time, sys

if "inference" in sys.modules:
    del sys.modules["inference"]

import inference

TEST_CASES = [
    ("weather — clean",          "What's the weather in Karachi in Celsius?",  [],            True),
    ("convert — clean",          "Convert 100 km to miles",                    [],            True),
    ("currency — clean",         "How much is 500 USD in EUR?",                [],            True),
    ("sql — clean",              "SELECT all users from users table",          [],            True),
    ("calendar — create",        f"Add a meeting on {D1}",                     [],            True),
    ("refusal — chitchat",       "Hey how are you?",                           [],            False),
    ("refusal — no tool",        "Send an email to John",                      [],            False),
    ("refusal — ambiguous",      "Convert that to euros",                      [],            False),
    ("adversarial — typo",       "whats the wether in dubaii",                 [],            True),
    ("adversarial — Urdu/EN",    "Mujhe London ka weather batao",              [],            True),
    ("multi-turn — resolve ref", "Now convert that to GBP",
        [{"role": "user", "content": "Convert 200 USD to EUR"},
         {"role": "assistant", "content": '<tool_call>{"tool":"currency","args":{"amount":200,"from":"USD","to":"EUR"}}</tool_call>'}],
        True),
]

passed, failed, latencies = 0, 0, []

for desc, prompt, history, expect_tool in TEST_CASES:
    t0 = time.time()
    result = inference.run(prompt, history)
    ms = (time.time() - t0) * 1000
    latencies.append(ms)

    has_tool = "<tool_call>" in result
    ok = (has_tool == expect_tool)
    status = "✅" if ok else "❌"
    if ok: passed += 1
    else:  failed += 1

    print(f"{status} [{ms:>5.0f}ms] {desc}")
    print(f"         → {result[:80]}")

avg_ms = sum(latencies) / len(latencies)
max_ms = max(latencies)
print(f"\n{'='*55}")
print(f"Score    : {passed}/{len(TEST_CASES)} passed")
print(f"Avg ms   : {avg_ms:.0f}ms  {'✅' if avg_ms <= 200 else '❌ OVER 200ms GATE'}")
print(f"Max ms   : {max_ms:.0f}ms")
print(f"{'='*55}")
if failed:
    print("\n⚠️  Failures? Try: more EPOCHS, more refusal examples, or check data formatting.")
else:
    print("\n✅ All tests passed. Ready to submit.")

## 🎨 Step 5 — Gradio Demo (required for submission)

## 🖥️ Gradio demo (interactive tool-calling playground)

**What this does**
- Launches a small web UI using `gradio` so you can chat with the fine-tuned model and see tool-calling behavior interactively.

**Why Gradio**
- It’s lightweight, works well in notebooks, and makes it easy to validate UX quickly without writing a full app.

**What to look for**
- The model should produce structured tool-call outputs in the expected schema.
- Try both “easy” prompts and edge cases (missing args, ambiguous requests) to see if the model asks clarifying questions.


In [ ]:
import gradio as gr
import sys

if "inference" not in sys.modules:
    import inference

def chat(user_message, history):
    """Gradio chat function. history = list of [user, assistant] pairs."""
    # Convert Gradio history format → inference.py format
    formatted_history = []
    for user_turn, assistant_turn in history:
        formatted_history.append({"role": "user",      "content": user_turn})
        formatted_history.append({"role": "assistant", "content": assistant_turn})

    result = inference.run(user_message, formatted_history)

    # Pretty-print tool calls
    import json, re
    match = re.search(r"<tool_call>(.*?)</tool_call>", result, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(1))
            display = (
                f"🔧 **Tool Called:** `{parsed['tool']}`\n\n"
                f"```json\n{json.dumps(parsed['args'], indent=2)}\n```"
            )
        except Exception:
            display = f"```\n{result}\n```"
    else:
        display = result

    return display

# ── Build UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title="Pocket-Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🤖 Pocket-Agent
    **On-device tool-calling assistant** — fully offline, quantized to ≤500 MB

    **Try:**
    - `What's the weather in Tokyo?`
    - `Convert 100 km to miles`
    - `How much is 500 USD in EUR?`
    - `Show all users from users table`
    - `Add a meeting tomorrow` (calendar)
    - `Tell me a joke` (should refuse)
    """)

    chatbot = gr.Chatbot(label="Pocket-Agent", height=400)
    msg     = gr.Textbox(label="Your message", placeholder="Ask something...", lines=1)
    with gr.Row():
        send  = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear")

    def respond(user_input, history):
        if not user_input.strip():
            return history, ""
        bot_reply = chat(user_input, history)
        history.append((user_input, bot_reply))
        return history, ""

    send.click(respond,  inputs=[msg, chatbot], outputs=[chatbot, msg])
    msg.submit(respond,  inputs=[msg, chatbot], outputs=[chatbot, msg])
    clear.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.launch(share=True)   # share=True gives a public URL for the demo video

## 📤 Step 6 — Submission Checklist

Before pushing to GitHub, verify every item:

| # | Check | Status |
|---|-------|--------|
| 1 | `inference.py` exists and `run()` works | ✅ |
| 2 | `model_q4.gguf` ≤ 500 MB | check above |
| 3 | `lora_adapter/` saved | ✅ |
| 4 | Gradio demo launches on CPU runtime | test it |
| 5 | No `import requests/urllib/http/socket` in inference.py | ✅ |
| 6 | `README.md` written with model choice, design decisions | write it |
| 7 | Avg latency ≤ 200ms on CPU | check self-test |
| 8 | Training data has zero overlap with public_test.jsonl | verify manually |

## 🔐 Git identity configuration (use with care)

**What this does**
- Sets Git author info via:
  - `git config --global user.email ...`
  - `git config --global user.name ...`

**Why it exists**
- Git requires an author identity to create commits.

**Important safety note (recommended)**
- `--global` modifies the *entire environment/user*, not just this project. In shared or ephemeral environments (like Colab), it can be surprising and may expose personal details in notebooks/logs.
- Prefer **repo-local** config instead:
  - `git config user.email "you@example.com"`
  - `git config user.name "Your Name"`

**If you’re not committing/pushing**
- You can skip this cell.


In [ ]:
!git config --global user.email "qudratullah0708@gmail.com"
!git config --global user.name "Qudrat"

## 🧭 Git branch naming alignment (`master` → `main`)

**What this does**
- Renames the local default branch from `master` to `main`.

**Why this value/command**
- Many GitHub repos default to `main`. Renaming avoids confusion and makes `git push -u origin main` work as expected.

**When to skip**
- If you’re not using Git/GitHub from this notebook, or your repo already uses `main`, you can skip.


In [ ]:
# Rename the local 'master' to 'main'
!git branch -M main

# Now try the push again
!git push -u origin main

## 🚀 (Optional) Step — Push artifacts to GitHub (with Git LFS)

**What this does**
- Sets up Git LFS (Large File Storage) so large model artifacts (e.g., `.bin`, `.safetensors`, `.gguf`) can be pushed without hitting normal Git size limits.

**Why it matters**
- Fine-tuned/quantized models often exceed GitHub’s regular file limits. Git LFS stores large binaries efficiently while keeping the repo usable.

**Notes / parameters**
- If you don’t plan to push to GitHub, you can skip this entire section.
- If you *do* push, consider adding a `.gitattributes` rule for `*.gguf filter=lfs diff=lfs merge=lfs -text` and similar patterns for checkpoints.


In [ ]:
# 1. Install LFS
!apt-get install git-lfs
!git lfs install

# 2. Tell Git to track GGUF files with LFS
!git lfs track "*.gguf"
!git add .gitattributes

# 3. Undo the previous commit so we can re-commit with LFS tracking
!git reset --soft HEAD~1

# 4. Re-add and re-commit
!git add .
!git commit -m "Pocket-Agent: fine-tuned Qwen2.5-1.5B with LFS for GGUF"

# 5. Push
!git push -u origin main